# Sensitivity Analysis — Are Your Subtypes Robust?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/07_sensitivity_analysis.ipynb)

**What this does:** Systematically varies analytical parameters (clustering algorithm, cluster count, normalization method, feature set) and measures whether subtypes remain consistent. High consistency = robust discovery; low consistency = parameter-dependent artifact.

**Key metric:** Adjusted Rand Index (ARI) between configurations. ARI = 1.0 means identical clusters; ARI ≈ 0 means random agreement.

**Prerequisites:** [00_quick_demo.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/00_quick_demo.ipynb)

In [ ]:
# Install pathway-subtyping
!pip install -q pathway-subtyping==0.3.1

import pathway_subtyping
print(f"pathway-subtyping v{pathway_subtyping.__version__}")

## 1. Generate Example Data

In [ ]:
from pathway_subtyping import SimulationConfig, generate_synthetic_data
import numpy as np

sim = generate_synthetic_data(SimulationConfig(
    n_samples=200,
    n_pathways=10,
    n_genes_per_pathway=20,
    n_subtypes=3,
    effect_size=1.2,
    noise_level=1.0,
    seed=42,
))

print(f"Data: {sim.pathway_scores.shape[0]} samples × {sim.pathway_scores.shape[1]} pathways")
print(f"Planted subtypes: {len(set(sim.true_labels))}")

## 2. Full Sensitivity Analysis

Run all four parameter axes in one call. This tests:

| Parameter | Variations | What It Tests |
|-----------|-----------|---------------|
| **Clustering algorithm** | GMM, K-means, Hierarchical, Spectral | Does the choice of algorithm matter? |
| **Number of clusters** | k = n-1 to n+2 | Is the cluster count stable? |
| **Normalization** | Z-score, min-max, robust, rank | Does preprocessing affect results? |
| **Feature subset** | Leave-one-out per pathway | Does any single pathway dominate? |

In [ ]:
from pathway_subtyping import run_sensitivity_analysis

result = run_sensitivity_analysis(
    pathway_scores=sim.pathway_scores,
    n_clusters=3,
    seed=42,
    robustness_threshold=0.7,
)

print(result.format_report())

## 3. Interpret Results

In [ ]:
print(f"Overall stability:        {result.overall_stability:.3f}")
print(f"Robust (>= 0.7):         {result.is_robust}")
print(f"Most sensitive parameter: {result.most_sensitive_parameter}")
print(f"Least sensitive parameter: {result.least_sensitive_parameter}")

print(f"\nPer-parameter breakdown:")
print(f"  {'Parameter':<25} {'Mean ARI':>10} {'Min ARI':>10}")
print(f"  {'-'*47}")
for name, pr in result.parameter_results.items():
    print(f"  {name:<25} {pr.mean_ari:>10.3f} {pr.min_ari:>10.3f}")

## 4. Test Individual Parameters

You can also test one parameter at a time for deeper investigation.

In [ ]:
from pathway_subtyping import vary_clustering_algorithm

# Which clustering algorithm gives the most consistent results?
algo_result = vary_clustering_algorithm(sim.pathway_scores, n_clusters=3, seed=42)

print(f"Algorithm Sensitivity (mean ARI: {algo_result.mean_ari:.3f})")
print(f"  Configurations tested: {algo_result.configurations}")
print(f"\n  Pairwise ARI:")
for pair, ari in algo_result.pairwise_ari.items():
    print(f"    {pair}: {ari:.3f}")

In [ ]:
from pathway_subtyping import vary_n_clusters

# How stable is the cluster count?
k_result = vary_n_clusters(sim.pathway_scores, cluster_range=(2, 6), seed=42)

print(f"Cluster Count Sensitivity (mean ARI: {k_result.mean_ari:.3f})")
print(f"  Configurations tested: {k_result.configurations}")
print(f"\n  Pairwise ARI:")
for pair, ari in k_result.pairwise_ari.items():
    print(f"    {pair}: {ari:.3f}")

In [ ]:
from pathway_subtyping import vary_feature_subset

# Does any single pathway dominate the clustering?
feat_result = vary_feature_subset(sim.pathway_scores, n_clusters=3, seed=42)

print(f"Feature Subset Sensitivity (mean ARI: {feat_result.mean_ari:.3f})")
print(f"  Configurations tested: {feat_result.configurations[:5]}...")
print(f"\n  Reference ARI (each config vs full data):")
for config, ari in sorted(feat_result.reference_ari.items(), key=lambda x: x[1]):
    print(f"    {config}: {ari:.3f}")

## 5. Visualize Sensitivity

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: per-parameter mean ARI
params = list(result.parameter_results.keys())
mean_aris = [result.parameter_results[p].mean_ari for p in params]
min_aris = [result.parameter_results[p].min_ari for p in params]

x = range(len(params))
axes[0].bar(x, mean_aris, color="steelblue", alpha=0.8, label="Mean ARI")
axes[0].bar(x, min_aris, color="salmon", alpha=0.6, label="Min ARI")
axes[0].axhline(y=0.7, color="green", linestyle="--", label="Robustness threshold")
axes[0].set_xticks(x)
axes[0].set_xticklabels([p.replace("_", "\n") for p in params], fontsize=9)
axes[0].set_ylabel("ARI")
axes[0].set_title("Sensitivity by Parameter")
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.05)

# Right: feature leave-one-out reference ARI
configs = list(feat_result.reference_ari.keys())
ref_aris = [feat_result.reference_ari[c] for c in configs]
colors = ["salmon" if a < 0.7 else "steelblue" for a in ref_aris]
axes[1].barh(range(len(configs)), ref_aris, color=colors, alpha=0.8)
axes[1].set_yticks(range(len(configs)))
axes[1].set_yticklabels(configs, fontsize=9)
axes[1].axvline(x=0.7, color="green", linestyle="--")
axes[1].set_xlabel("ARI vs Full Data")
axes[1].set_title("Feature Leave-One-Out")
axes[1].set_xlim(0, 1.05)

plt.suptitle("Sensitivity Analysis Results", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Interpretation Guide

| Overall Stability | Meaning | Action |
|------------------|---------|--------|
| > 0.8 | Highly robust | Subtypes are reliable |
| 0.7 - 0.8 | Robust | Acceptable for most analyses |
| 0.5 - 0.7 | Moderate | Investigate the most sensitive parameter |
| < 0.5 | Unstable | Subtypes may be artifacts — reconsider approach |

**If a specific parameter is sensitive:**
- **Clustering algorithm:** Try ensemble approaches or use the most stable algorithm
- **Number of clusters:** Use `select_n_clusters()` for data-driven K selection
- **Normalization:** Standardize your preprocessing across analyses
- **Feature subset:** A dominant pathway may need removal or the pathway set may be too small

## Next Steps

- **Characterization:** [08_characterization.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/08_characterization.ipynb) — interpret what makes each subtype distinct
- **API reference:** [Sensitivity API](https://github.com/topmist-admin/pathway-subtyping-framework/blob/main/docs/api/sensitivity.md)

---
*Built with [pathway-subtyping](https://pypi.org/project/pathway-subtyping/). Disease-agnostic. Open source.*